# FastAPI App
Notebook version of `Service/app.py`. Running the code cells creates the FastAPI `app`, receiver, and detector.

In [ ]:
from pathlib import Path
NB_DIR = Path.cwd() if Path.cwd().name == "Service_jupyter" else Path.cwd() / "Service_jupyter"
BACKEND_DIR = NB_DIR.parent
get_ipython().run_line_magic("run", str(NB_DIR / "detector.ipynb"))

In [ ]:
from contextlib import asynccontextmanager
from typing import Any

from fastapi import FastAPI, Query
from fastapi.responses import FileResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

STATIC_DIR = BACKEND_DIR / "Service" / "static"

receiver = MuseReceiver(DEFAULT_ADDRESS)
detector = EmotionDetectionService(receiver)

class ConnectRequest(BaseModel):
    address: str | None = None

@asynccontextmanager
async def lifespan(_: FastAPI):
    yield
    detector.stop()
    receiver.stop()

app = FastAPI(
    title="Muse EEG Emotion Recognition API",
    description="Connect to a Muse headset and expose realtime emotion predictions.",
    version="1.0.0",
    lifespan=lifespan,
)
app.mount("/static", StaticFiles(directory=str(STATIC_DIR)), name="static")

@app.get("/", include_in_schema=False)
def index() -> FileResponse:
    return FileResponse(STATIC_DIR / "index.html")

@app.get("/api/status")
def status() -> dict[str, Any]:
    return {"receiver": receiver.snapshot(), "detector": detector.status()}

@app.get("/api/eeg-waves")
def eeg_waves(
    seconds: float = Query(default=4.0, ge=1.0, le=10.0),
    points: int = Query(default=420, ge=64, le=1024),
) -> dict[str, Any]:
    return receiver.waveform_snapshot(int(seconds * MUSE_SAMPLE_RATE), points)

@app.post("/api/connect")
def connect(payload: ConnectRequest | None = None) -> dict[str, Any]:
    requested_address = payload.address if payload else None
    address = (requested_address or DEFAULT_ADDRESS).strip() or DEFAULT_ADDRESS
    receiver.start(address)
    return {"ok": True, "message": "Connection started.", "status": receiver.snapshot()}

@app.post("/api/disconnect")
def disconnect() -> dict[str, Any]:
    detector.stop()
    receiver.stop()
    return {"ok": True, "message": "Disconnected.", "status": receiver.snapshot()}

@app.post("/api/start-detection")
def start_detection() -> Any:
    if not receiver.running:
        return JSONResponse(status_code=400, content={"ok": False, "message": "Connect the Muse device first."})
    detector.start()
    return {"ok": True, "message": "Detection started.", "status": detector.status()}

@app.post("/api/stop-detection")
def stop_detection() -> dict[str, Any]:
    detector.stop()
    return {"ok": True, "message": "Detection stopped.", "status": detector.status()}

print("FastAPI app created.")